# trlm training on Colab
Upload `trlm_colab.zip` (from `scripts/pack_for_colab.sh`) to `MyDrive/trlm/`. Runtime: GPU. Every run resumes from Drive after a disconnect; just re-run all cells.

In [ ]:
!pip -q install tokenizers pyarrow pyyaml
import torch; print(torch.__version__, torch.cuda.get_device_name(0))

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
ROOT = '/content/drive/MyDrive/trlm'
!mkdir -p $ROOT/models
!cd /content && rm -rf trlm && mkdir trlm && cd trlm && unzip -q $ROOT/trlm_colab.zip
%cd /content/trlm

In [ ]:
# Two-minute throughput test (AGENTS.md section 5)
!python -m src.train.throughput --device cuda --variant mlp --seconds 120 --amp | tee -a $ROOT/throughput_colab.jsonl
!python -m src.train.throughput --device cuda --variant attn --seconds 120 --amp | tee -a $ROOT/throughput_colab.jsonl

In [ ]:
# Pilot
!python -m src.train.runs pilot_lr --root $ROOT/models --amp
!python -m src.train.runs pilot_budget --root $ROOT/models --amp
!python -m src.train.runs pilot_report --root $ROOT/models && cp logs/pilot.md $ROOT/pilot.md

In [ ]:
# Final 10 runs: fill in budget and LR from pilot.md after human go-ahead
BUDGET = None
LR = None
assert BUDGET and LR
!python -m src.train.runs final --root $ROOT/models --amp --budget_tokens $BUDGET --peak_lr $LR